In [1]:
import pyroomacoustics as pra

import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.autograd import profiler
import torchaudio
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore


from einops import rearrange

from src.dataset import SignalDataset, TRUNetDataset
from src.loss import loss_tot, loss_MR, loss_MR_w
from models.fspen import * # FullSubPathExtension, FullSubPathExtension_3_heads, FullSubPathExtension_ver2, FullSubPathExtension_abs_pha, FullSubPathExtension_abs_pha_mapping, FullSubPathExtension_ver2_abs_pha, FullSubPathExtension_ver3

from IPython.display import Audio

from src.utils import model_eval, model_eval_fspen2x_ver3, model_eval_3_heads, use_pcs, inv_pcs, model_eval_old

import matplotlib.pyplot as plt

In [2]:
TEST_DIR = os.path.join("data", "DS_10283_2791", "clean_testset_wav")
TEST_NOISE_DIR = os.path.join("data", "DS_10283_2791", "noisy_testset_wav")
NOISE_DIR = os.path.join("data", "demand_test")

CHKP_DIR = "checkpoints"

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)

gen = torch.Generator()
gen.manual_seed(SEED)

In [4]:
from src.fspen_configs import *

configs = TrainConfig_48khz() # TrainConfig_explicit_unfold()
# print(sum(configs.bands_num_in_groups), configs.dual_path_extension["num_modules"])
fspen = FullSubPathExtension(configs=configs)# .to(DEVICE)

state_d = torch.load(os.path.join(CHKP_DIR, "fspen_chkp", "TrainConfig_48khz_baseline#0.pt"), map_location="cpu",  weights_only=False)

In [5]:
fspen.load_state_dict(state_d["model_state_dict"])

<All keys matched successfully>

In [6]:
# N_FFTS = 512
# HOP_LENGTH = 256
# HID_SIZE = 32
# SR = 16_000

N_FFTS = configs.n_fft
HOP_LENGTH = configs.hop_length
HID_SIZE = 64
SR = configs.sample_rate
BATCH_SIZE = 8 # 32

DEVICE = "cpu" # torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cpu time!!!


In [7]:
rir_dict = {1: os.path.join("data", "rirs48_small_3_test"), 1: os.path.join("data", "rirs48_medium_3_test"), 1: os.path.join("data", "rirs48_large_3_test"), 1: os.path.join("data", "rirs48_super_large_3_test")}
dataset = TRUNetDataset(TEST_DIR, sr=SR, noise_dir=NOISE_DIR, rir_dir=rir_dict, snr=[0, 5, 10, 15], rir_proba=0.85, noise_proba=0.85, rir_target=False, return_noise=False, return_rir=False)
dataset.set_epoch(99)

36
12


In [8]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [9]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [10]:
def pad_sequence(batch):
    if not batch:
        return torch.zeros(0), torch.zeros(0)

    input_signal, target_signal, noise, rir = zip(*batch)
        
    max_len_s = max(s.shape[-1] for s in input_signal)
    
    padded_input = torch.zeros(len(input_signal), max_len_s)
    padded_target = torch.zeros(len(target_signal), max_len_s)
    
    for i, s in enumerate(input_signal):
        padded_input[i, :s.shape[-1]] = s
        padded_target[i, :s.shape[-1]] = target_signal[i]

    return padded_input, padded_target


def collate_fn(batch):
    
    padded_input, padded_target = pad_sequence(batch)
        
    padded_input = padded_input.reshape(-1, padded_input.shape[-1])
    padded_target = padded_target.reshape(-1, padded_input.shape[-1])

    return padded_input, padded_target

In [11]:
test_dataloader = DataLoader(dataset, batch_size=1, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [12]:
import time

def check_stream_inference(model, loader, window_size = 1 * SR // 4, device="cpu"):
    model.eval()

    result_nisqa_full = []
    result_rtf_full = []
    result_nisqa_chunk = []
    result_rtf_chunk = []
    with torch.no_grad():
        for signal, target in tqdm(loader):
            signal = signal.to(device)
            target = target.to(device)
            window = vorbis_window(N_FFTS).to(device)
    
            start_time = time.time()
            spec = torch.stft(
                signal,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            ) 

            # spec = use_pcs(spec, N_FFTS)
            
            output, _ = model_eval_old(model, spec, configs, device, hid_size=HID_SIZE)

            # output = inv_pcs(output.abs(), output.angle())

            window = vorbis_window(N_FFTS).to(device)
            output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                   window=window,
                                   # onesided=True,
                                   return_complex=False,
                                   normalized=True,
                                   center=True)
            
            end_time = time.time()
            
            result_rtf_full.append((signal.shape[-1] / SR) / (end_time - start_time))
            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            result_nisqa_full.append(nisqa_score)

            for j in range(0, signal.shape[-1], window_size):
                chunk = signal[..., j:j+window_size]
                
                if chunk.shape[-1] < 10_000:
                    continue

                start_time = time.time()
                spec = torch.stft(
                    chunk,
                    n_fft=N_FFTS,
                    hop_length=HOP_LENGTH,
                    # onesided=True,
                    win_length=N_FFTS,
                    window=window,
                    return_complex=True,
                    normalized=True,
                    center=True
                )

                output, _ = model_eval_old(model, spec, configs, device, hid_size=HID_SIZE)

                window = vorbis_window(N_FFTS).to(device)
                output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                    window=window,
                                    # onesided=True,
                                    return_complex=False,
                                    normalized=True,
                                    center=True)
                
                end_time = time.time()


                result_rtf_chunk.append((chunk.shape[-1] / SR) / (end_time - start_time))
                # print(output.shape)
                nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

                result_nisqa_chunk.append(nisqa_score)
                

    print(f"Mean nisqa for full audio: ", torch.stack(result_nisqa_full).mean(dim=0))
    print(f"Mean rtf for full audio: ", torch.tensor(result_rtf_full).mean(dim=0), 1 / torch.tensor(result_rtf_full).mean(dim=0))
    print("---" * 10)
    print("Mean nisqa for \"stream\" audio: ", torch.stack(result_nisqa_chunk).mean(dim=0))
    print("Mean rtf for \"stream\" audio: ", torch.tensor(result_rtf_chunk).mean(dim=-1), 1 / torch.tensor(result_rtf_chunk).mean(dim=-1))

    return result_nisqa_full, result_rtf_full, result_nisqa_chunk, result_rtf_chunk

In [13]:
# _ = check_stream_inference(fspen, test_dataloader, device="cpu")

In [14]:
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torch_stoi import NegSTOILoss

srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to("cuda")
stoi = NegSTOILoss(SR, use_vad=False, do_resample=False).to("cuda")
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [15]:
from torchaudio.transforms import Resample
from thop import profile


def get_metrics(model, loader, device="cpu"):
    model.eval()
    
    model = model.to(device)
    
    input_nisqa_scores = []
    input_pesq_scores = []
    input_stoi_scores = []
    input_srmr_scores = []
    input_dnsmos_scores = []
    
    gt_nisqa_scores = []
    gt_pesq_scores = []
    gt_stoi_scores = []
    gt_srmr_scores = []
    gt_dnsmos_scores = []

    with torch.no_grad():
        for signal, target in tqdm(loader):
            signal = signal.to(device)
            target = target.to(device)
            window = vorbis_window(N_FFTS).to(device)
            
            # min_l = min(output.shape[-1], signal.shape[-1])
            input_nisqa_score, _, _ = process(signal.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            gt_nisqa_score, _, _ = process(target.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

            resampler = Resample(SR, 16_000)
            signal = resampler(signal.cpu()).cuda()
            target = resampler(target.cpu()).cuda()

            input_stoi_score = stoi(signal, target)
            gt_stoi_score = stoi(target, target)

            input_srmr_score = srmr(signal.detach().cpu())
            gt_srmr_score = srmr(target.detach().cpu())
            input_dnsmos_score = dnsmos(signal.detach())
            gt_dnsmos_score = dnsmos(target.detach())

            try:
                input_pesq_score = pesq(signal, target)
                gt_pesq_score = pesq(target, target)
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            input_nisqa_scores.append(input_nisqa_score[0])
            input_pesq_scores.append(input_pesq_score)
            input_stoi_scores.append(input_stoi_score.cpu())
            input_srmr_scores.append(input_srmr_score.cpu())
            input_dnsmos_scores.append(input_dnsmos_score.cpu())

            gt_nisqa_scores.append(gt_nisqa_score[0])
            gt_pesq_scores.append(gt_pesq_score)
            gt_stoi_scores.append(gt_stoi_score.cpu())
            gt_srmr_scores.append(gt_srmr_score.cpu())
            gt_dnsmos_scores.append(gt_dnsmos_score.cpu())

    result = {"nisqa": input_nisqa_scores, "stoi": input_stoi_scores, "srmr": input_srmr_scores, "pesq": input_pesq_scores, "dnsmos": input_dnsmos_scores}
    result_gt = {"nisqa": gt_nisqa_scores, "stoi": gt_stoi_scores, "srmr": gt_srmr_scores, "pesq": gt_pesq_scores, "dnsmos": gt_dnsmos_scores}

    return result, result_gt

In [16]:
metrics, metrics_gt = get_metrics(fspen, test_dataloader, device="cuda")

100%|██████████| 824/824 [21:59<00:00,  1.60s/it]


In [17]:
print("NISQA:", torch.vstack(metrics["nisqa"]).mean(dim=0))
print("PESQ:", torch.vstack(metrics["pesq"]).mean(dim=0))
print("SRMR:", torch.vstack(metrics["srmr"]).mean(dim=0))
print("STOI:", -torch.vstack(metrics["stoi"]).mean(dim=0))
print("DNSMOS:", torch.vstack(metrics["dnsmos"]).mean(dim=0))

NISQA: tensor([2.730, 2.400, 3.458, 3.459, 3.348])
PESQ: tensor([1.681], device='cuda:0')
SRMR: tensor([6.644])
STOI: tensor([0.823])
DNSMOS: tensor([2.980, 3.135, 2.692, 2.364], dtype=torch.float64)


In [18]:
print("NISQA:", torch.vstack(metrics_gt["nisqa"]).mean(dim=0))
print("PESQ:", torch.vstack(metrics_gt["pesq"]).mean(dim=0))
print("SRMR:", torch.vstack(metrics_gt["srmr"]).mean(dim=0))
print("STOI:", -torch.vstack(metrics_gt["stoi"]).mean(dim=0))
print("DNSMOS:", torch.vstack(metrics_gt["dnsmos"]).mean(dim=0))

NISQA: tensor([4.071, 4.118, 4.011, 4.141, 4.133])
PESQ: tensor([4.644], device='cuda:0')
SRMR: tensor([8.943])
STOI: tensor([1.])
DNSMOS: tensor([3.557, 3.464, 3.963, 3.153], dtype=torch.float64)


NISQA: tensor([3.737, 3.810, 3.801, 3.787, 3.884])
PESQ: tensor([2.413])
SRMR: tensor([10.142])
STOI: tensor([0.885])
DNSMOS: tensor([3.194, 2.998, 3.770, 2.667], dtype=torch.float64)
MACs: 529545345.50125134